In [2]:
"""
Shapes-based pathways construction from GTFS and OSM graph.
Generates walking connections between routes using GTFS shapes corridors rather than stops.
Output columns:
- start_route_id, end_route_id, walking_distance_m, walking_path_nodes
Optionally: nearest shape point ids for reference.
"""
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from pathlib import Path

# Configure paths
GTFS_DIR = Path(r"C:/Users/ahmed/Downloads/gtfsAlex")
GRAPH_XML = Path(r"C:/Users/ahmed/Documents/grad/draft1/labeled.osm")

# Load GTFS
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")

print(f"Loaded routes={len(routes)}, trips={len(trips)}, shapes points={len(shapes)}")

# Load OSM graph (walkable)
g = ox.graph_from_xml(filepath=str(GRAPH_XML), bidirectional=True)
g = ox.convert.to_undirected(g)
print(f"Graph loaded with {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


Loaded routes=104, trips=192, shapes points=29358
Graph loaded with 45784 nodes, 65854 edges


In [9]:
# Map routes to representative shapes
# Choose for each route the shape_id with the longest shape (most points).

# trip -> route, trip -> shape
trip_to_route = pd.Series(trips.route_id.values, index=trips.trip_id).to_dict()
trip_to_shape = pd.Series(trips.shape_id.values, index=trips.trip_id).to_dict()

shape_counts = (
    trips.dropna(subset=['shape_id'])
         .groupby(['route_id','shape_id']).size().reset_index(name='n_trips')
)

# For each (route_id, shape_id), compute number of points in shapes.txt
shape_points_count = shapes.groupby('shape_id').size().rename('n_points').reset_index()
route_shape_points = shape_counts.merge(shape_points_count, on='shape_id', how='left')

# Pick the shape_id with most points per route
rep_shape_per_route = route_shape_points.sort_values(['route_id','n_points'], ascending=[True, False]).drop_duplicates('route_id')
route_to_shape = pd.Series(rep_shape_per_route.shape_id.values, index=rep_shape_per_route.route_id).to_dict()

print(f"Representative shapes mapped for {len(route_to_shape)} routes")

# Build ordered shape points per representative shape
ordered_shapes = (
    shapes[shapes.shape_id.isin(rep_shape_per_route['shape_id'])]
    .sort_values(['shape_id','shape_pt_sequence'])
)

# Convenience dict: route -> list[(lat, lon, seq)]
route_to_shape_points = {}
for r, sid in route_to_shape.items():
    sdf = ordered_shapes[ordered_shapes.shape_id == sid]
    pts = list(zip(sdf['shape_pt_lat'].values, sdf['shape_pt_lon'].values, sdf['shape_pt_sequence'].values))
    route_to_shape_points[r] = pts

# Define start/end shape nodes per route as first/last points
route_to_shape_start = {r: (lat, lon) for r, (lat, lon, _) in ((r, pts[0]) for r, pts in route_to_shape_points.items() if len(pts)>0)}
route_to_shape_end = {r: (lat, lon) for r, (lat, lon, _) in ((r, pts[-1]) for r, pts in route_to_shape_points.items() if len(pts)>0)}


Representative shapes mapped for 104 routes


In [10]:
# Map shape points to nearest graph nodes (vectorized per route)
import numpy as np

route_to_shape_nodes = {}
for r, pts in route_to_shape_points.items():
    if not pts:
        route_to_shape_nodes[r] = []
        continue
    lats = np.array([lat for lat, lon, _ in pts])
    lons = np.array([lon for lat, lon, _ in pts])
    nodes = ox.distance.nearest_nodes(g, X=lons, Y=lats)
    route_to_shape_nodes[r] = list(nodes)

# Start/end nodes from shapes
route_to_shape_start_node = {}
route_to_shape_end_node = {}
for r, nodes in route_to_shape_nodes.items():
    if nodes:
        route_to_shape_start_node[r] = nodes[0]
        route_to_shape_end_node[r] = nodes[-1]

print(f"Mapped shape points to nodes for {len(route_to_shape_nodes)} routes")


Mapped shape points to nodes for 104 routes


In [11]:
# Proximity via shapes corridors: r2 start near r1 corridor, r3 end near r1 corridor
from math import radians, sin, cos, asin, sqrt

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

node_x = nx.get_node_attributes(g, 'x')
node_y = nx.get_node_attributes(g, 'y')
node_latlon = {n: (node_y.get(n), node_x.get(n)) for n in g.nodes()}

NEAR_RADIUS_M = 300

proximity_candidates = []  # (start_route_id, end_route_id, start_node, end_node)
route_ids = list(route_to_shape_nodes.keys())

for r1 in route_ids:
    r1_nodes = route_to_shape_nodes.get(r1, [])
    if not r1_nodes:
        continue

    # r2 starts near r1
    for r2, s_node in route_to_shape_start_node.items():
        if r2 == r1:
            continue
        lat2, lon2 = node_latlon.get(s_node, (None, None))
        if lat2 is None:
            continue
        near = False
        closest_r1_node = None
        closest_dist = float('inf')
        for n in r1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat2, lon2)
            if d < closest_dist:
                closest_dist = d
                closest_r1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            proximity_candidates.append((r2, r1, s_node, closest_r1_node))

    # r3 ends near r1
    for r3, e_node in route_to_shape_end_node.items():
        if r3 == r1:
            continue
        lat3, lon3 = node_latlon.get(e_node, (None, None))
        if lat3 is None:
            continue
        near = False
        closest_r1_node = None
        closest_dist = float('inf')
        for n in r1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat3, lon3)
            if d < closest_dist:
                closest_dist = d
                closest_r1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            proximity_candidates.append((r3, r1, e_node, closest_r1_node))

print(f"Proximity candidates: {len(proximity_candidates)}")


Proximity candidates: 3011


In [12]:
# Compute walking shortest paths for shape-based candidates and assemble DataFrame
import math

if not nx.get_edge_attributes(g, 'length'):
    g = ox.distance.add_edge_lengths(g)

records = []
failed = 0
for start_route_id, end_route_id, start_node, end_node in proximity_candidates:
    try:
        path = nx.shortest_path(g, source=start_node, target=end_node, weight='length')
        dist = 0.0
        for u, v in zip(path[:-1], path[1:]):
            data = g.get_edge_data(u, v)
            if not data:
                continue
            ed = next(iter(data.values()))
            dist += float(ed.get('length', 1.0))
        records.append({
            'start_route_id': start_route_id,
            'end_route_id': end_route_id,
            'walking_distance_m': dist,
            'walking_path_nodes': path,
        })
    except Exception as e:
        failed += 1

print(f"Built {len(records)} shape-based pathways, failed={failed}")
shapes_pathways_df = pd.DataFrame.from_records(records).drop_duplicates(subset=['start_route_id','end_route_id']).reset_index(drop=True)
shapes_pathways_df.head()


Built 3011 shape-based pathways, failed=0


,start_route_id,end_route_id,walking_distance_m,walking_path_nodes
0,-Z1R0bmP-3IQrktLaw1-i,-H9LP4vuOqj-RIRCZNg_6,113.466015,"[1886821074, 1886821204, 11462532250]"
1,RSdPdtwiGzlvedMSUpA36,-H9LP4vuOqj-RIRCZNg_6,113.466015,"[1886821074, 1886821204, 11462532250]"
2,d1tk5YD606wPnGF4CLm4i,-H9LP4vuOqj-RIRCZNg_6,0.000000,[11462532250]
3,lbr0LeYaqC-PO2eSd5dsH,-H9LP4vuOqj-RIRCZNg_6,446.845175,"[265746704, 4326497482, 6942587501, 4326497475..."
4,50n7_gqFiIrgeNHtVzwF0,-H9LP4vuOqj-RIRCZNg_6,90.518506,"[1886821204, 11462532250]"


In [13]:
# Save CSV and plotting helper for two routes + walking path
from itertools import pairwise
import folium

output_csv = Path("shapes_pathways.csv")
shapes_pathways_df.to_csv(output_csv, index=False)
print(f"Saved shapes pathways to {output_csv.resolve()}")

# Build route polyline from shapes (using the representative shape points)
def _shape_route_coords(G, route_id):
    nodes = route_to_shape_nodes.get(route_id, [])
    if len(nodes) < 2:
        return []
    # ensure lengths
    if not nx.get_edge_attributes(G, 'length'):
        ox.distance.add_edge_lengths(G)
    coords = []
    for u, v in pairwise(nodes):
        try:
            sp = nx.shortest_path(G, u, v, weight='length')
            coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in sp if 'x' in G.nodes[n] and 'y' in G.nodes[n]])
        except Exception:
            continue
    return coords


def plot_shapes_route_pair_with_path(start_route_id, end_route_id, save_html=None, map_tiles="cartodbpositron"):
    row = shapes_pathways_df[(shapes_pathways_df['start_route_id'] == start_route_id) & (shapes_pathways_df['end_route_id'] == end_route_id)]
    if row.empty:
        raise ValueError("No pathway found for the given route pair in shapes_pathways_df")
    row = row.iloc[0]
    walking_nodes = row['walking_path_nodes']

    # center
    mid_n = walking_nodes[len(walking_nodes)//2] if len(walking_nodes) > 0 else route_to_shape_nodes.get(start_route_id, [None])[0]
    center_lat = g.nodes[mid_n]['y']
    center_lon = g.nodes[mid_n]['x']
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    r1_coords = _shape_route_coords(g, start_route_id)
    r2_coords = _shape_route_coords(g, end_route_id)

    if r1_coords:
        folium.PolyLine(r1_coords, color="#1f77b4", weight=5, opacity=0.8, tooltip=f"Route {start_route_id}").add_to(m)
    if r2_coords:
        folium.PolyLine(r2_coords, color="#ff7f0e", weight=5, opacity=0.8, tooltip=f"Route {end_route_id}").add_to(m)

    walk_coords = [(g.nodes[n]['y'], g.nodes[n]['x']) for n in walking_nodes if 'x' in g.nodes[n] and 'y' in g.nodes[n]]
    if walk_coords:
        folium.PolyLine(walk_coords, color="#2ca02c", weight=6, opacity=0.9, tooltip=f"Walk {len(walk_coords)} pts").add_to(m)

    if save_html:
        m.save(save_html)
    return m


Saved shapes pathways to C:\Users\ahmed\Documents\grad\draft1\shapes_pathways.csv


In [14]:
# Shape-level pipeline (ignore routes) — per shape_id
# Build per-shape ordered sequences and start/end points

ordered_shapes_all = shapes.sort_values(['shape_id','shape_pt_sequence']).copy()
shape_ids = ordered_shapes_all['shape_id'].unique().tolist()

# Dicts: shape_id -> list[(lat, lon, seq)]
shape_to_points = {}
for sid, sdf in ordered_shapes_all.groupby('shape_id'):
    pts = list(zip(sdf['shape_pt_lat'].values, sdf['shape_pt_lon'].values, sdf['shape_pt_sequence'].values))
    if len(pts) >= 2:
        shape_to_points[sid] = pts

print(f"Prepared {len(shape_to_points)} shapes with >=2 points")


Prepared 192 shapes with >=2 points


In [15]:
# Map per-shape points to nearest nodes
import numpy as np

shape_to_nodes = {}
for sid, pts in shape_to_points.items():
    lats = np.array([lat for lat, lon, _ in pts])
    lons = np.array([lon for lat, lon, _ in pts])
    nodes = ox.distance.nearest_nodes(g, X=lons, Y=lats)
    shape_to_nodes[sid] = list(nodes)

shape_to_start_node = {sid: nodes[0] for sid, nodes in shape_to_nodes.items() if len(nodes) > 0}
shape_to_end_node = {sid: nodes[-1] for sid, nodes in shape_to_nodes.items() if len(nodes) > 0}

print(f"Mapped nodes for {len(shape_to_nodes)} shapes")


Mapped nodes for 192 shapes


In [16]:
# Proximity candidates between shapes: s2 start near s1 corridor, s3 end near s1 corridor
from math import radians, sin, cos, asin, sqrt

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

node_x = nx.get_node_attributes(g, 'x')
node_y = nx.get_node_attributes(g, 'y')
node_latlon = {n: (node_y.get(n), node_x.get(n)) for n in g.nodes()}

NEAR_RADIUS_M = 300

shape_proximity_candidates = []  # (start_shape_id, end_shape_id, start_node, end_node)
shape_ids = list(shape_to_nodes.keys())

for s1 in shape_ids:
    s1_nodes = shape_to_nodes.get(s1, [])
    if not s1_nodes:
        continue

    # s2 starts near s1
    for s2, s_node in shape_to_start_node.items():
        if s2 == s1:
            continue
        lat2, lon2 = node_latlon.get(s_node, (None, None))
        if lat2 is None:
            continue
        near = False
        closest_s1_node = None
        closest_dist = float('inf')
        for n in s1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat2, lon2)
            if d < closest_dist:
                closest_dist = d
                closest_s1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            shape_proximity_candidates.append((s2, s1, s_node, closest_s1_node))

    # s3 ends near s1
    for s3, e_node in shape_to_end_node.items():
        if s3 == s1:
            continue
        lat3, lon3 = node_latlon.get(e_node, (None, None))
        if lat3 is None:
            continue
        near = False
        closest_s1_node = None
        closest_dist = float('inf')
        for n in s1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat3, lon3)
            if d < closest_dist:
                closest_dist = d
                closest_s1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            shape_proximity_candidates.append((s3, s1, e_node, closest_s1_node))

print(f"Shape-level proximity candidates: {len(shape_proximity_candidates)}")


Shape-level proximity candidates: 10266


In [25]:
# Build shape-level pathways via shortest paths and save + plot helper
import folium
from itertools import pairwise

if not nx.get_edge_attributes(g, 'length'):
    g = ox.distance.add_edge_lengths(g)

shape_records = []
failed = 0
for start_shape, end_shape, start_node, end_node in shape_proximity_candidates:
    try:
        sp_nodes = nx.shortest_path(g, source=start_node, target=end_node, weight='length')
        dist = 0.0
        for u, v in pairwise(sp_nodes):
            data = g.get_edge_data(u, v)
            if not data:
                continue
            ed = next(iter(data.values()))
            dist += float(ed.get('length', 1.0))
        shape_records.append({
            'start_shape_id': start_shape,
            'end_shape_id': end_shape,
            'walking_distance_m': dist,
            'walking_path_nodes': sp_nodes,
        })
    except Exception:
        failed += 1

print(f"Built {len(shape_records)} shape-level pathways, failed={failed}")
shape_level_pathways_df = pd.DataFrame.from_records(shape_records).drop_duplicates(subset=['start_shape_id','end_shape_id']).reset_index(drop=True)

# Save
out_csv = Path('shape_level_pathways.csv')
shape_level_pathways_df.to_csv(out_csv, index=False)
print(f"Saved {out_csv.resolve()}")

# Plotting function for two shape_ids
def plot_shape_pair_with_path(start_shape_id, end_shape_id, save_html=None, map_tiles="cartodbpositron"):
    row = shape_level_pathways_df[(shape_level_pathways_df['start_shape_id'] == start_shape_id) & (shape_level_pathways_df['end_shape_id'] == end_shape_id)]
    if row.empty:
        raise ValueError("No pathway found for the given shape_id pair")
    row = row.iloc[0]
    walking_nodes = row['walking_path_nodes']

    mid_n = walking_nodes[len(walking_nodes)//2] if len(walking_nodes) > 0 else shape_to_nodes.get(start_shape_id, [None])[0]
    center_lat = g.nodes[mid_n]['y']
    center_lon = g.nodes[mid_n]['x']
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    # draw shapes polylines (approx via routing between consecutive shape nodes)
    def _coords_for_shape(sid):
        nodes = shape_to_nodes.get(sid, [])
        coords = []
        for u, v in pairwise(nodes):
            try:
                sp = nx.shortest_path(g, u, v, weight='length')
                coords.extend([(g.nodes[n]['y'], g.nodes[n]['x']) for n in sp if 'x' in g.nodes[n] and 'y' in g.nodes[n]])
            except Exception:
                continue
        return coords

    c1 = _coords_for_shape(start_shape_id)
    c2 = _coords_for_shape(end_shape_id)

    if c1:
        folium.PolyLine(c1, color="#1f77b4", weight=5, opacity=0.85, tooltip=f"shape {start_shape_id}").add_to(m)
    if c2:
        folium.PolyLine(c2, color="#ff7f0e", weight=5, opacity=0.85, tooltip=f"shape {end_shape_id}").add_to(m)

    wcoords = [(g.nodes[n]['y'], g.nodes[n]['x']) for n in walking_nodes if 'x' in g.nodes[n] and 'y' in g.nodes[n]]
    if wcoords:
        folium.PolyLine(wcoords, color="#2ca02c", weight=6, opacity=0.95, tooltip=f"walk {len(wcoords)} pts").add_to(m)

    if save_html:
        m.save(save_html)
    return m


Built 10266 shape-level pathways, failed=0
Saved C:\Users\ahmed\Documents\grad\draft1\shape_level_pathways.csv


In [27]:
# Enrich shape-level pathways with trip names (headsigns) and re-save
# For each shape_id, aggregate unique trip_headsigns from trips.txt

# Some feeds may not have trip_headsign; handle missing by falling back to trip_id
trip_names_by_shape = (
    trips.assign(trip_headsign=trips.get('trip_headsign'))
         .groupby('shape_id')['trip_headsign']
         .apply(lambda s: sorted({str(x) for x in s.dropna().astype(str)}) if s.notna().any() else [])
         .to_dict()
)
trip_ids_by_shape = (
    trips.groupby('shape_id')['trip_id']
         .apply(lambda s: sorted({str(x) for x in s.astype(str)}))
         .to_dict()
)

def names_for_shape(shape_id):
    names = trip_names_by_shape.get(shape_id) or []
    if names:
        return names
    # fallback to trip ids if no headsigns
    return trip_ids_by_shape.get(shape_id, [])

shape_level_pathways_df['start_trip_names'] = shape_level_pathways_df['start_shape_id'].map(names_for_shape)
shape_level_pathways_df['end_trip_names'] = shape_level_pathways_df['end_shape_id'].map(names_for_shape)

# Re-save CSV with the new columns
out_csv = Path('shape_level_pathways.csv')
shape_level_pathways_df.to_csv(out_csv, index=False)
print(f"Updated {out_csv.resolve()} with trip names columns: start_trip_names, end_trip_names")

# Show a few rows
shape_level_pathways_df.head(10)


Updated C:\Users\ahmed\Documents\grad\draft1\shape_level_pathways.csv with trip names columns: start_trip_names, end_trip_names


,start_shape_id,end_shape_id,walking_distance_m,walking_path_nodes,start_trip_names,end_trip_names
0,0eMropKVlaFEwtKn18KiF_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,0.000000,[6487833999],[El-Sa'ah (Clock Square)],[Bakus]
1,1v5hAqd3R5NHO2_mRDJ3z_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1052.347688,"[317908405, 7041564226, 7041564225, 7041564199...",[Al-Maraghi],[Bakus]
2,3SynmEGmJSDoBYRTRgEkj_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,202.981279,"[5275557217, 6991754699, 1128476936]",[El-Mansheya],[Bakus]
3,7PVy-xzsAclt9bvyLn2D__Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1060.823750,"[9368718200, 7041564226, 7041564225, 704156419...",[Asafra],[Bakus]
4,9iBvMMprqaal91Eesle9O_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1737.478490,"[7043828811, 7043828810, 9414993328, 702373236...",[Train Station (El-Shohada Square)],[Bakus]
5,Da9EIaUt19ZSMI3qwGvzZ_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1737.478490,"[7043828811, 7043828810, 9414993328, 702373236...",[Sidi Gabir],[Bakus]
6,K63Xp_QNyqIIHRQrSOVNB_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1052.347688,"[317908405, 7041564226, 7041564225, 7041564199...",[El-Sa'ah (Clock Square)],[Bakus]
7,TIF4utcDqnWmdbkuyp9Dq_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1052.347688,"[317908405, 7041564226, 7041564225, 7041564199...",[El-Sa'ah (Clock Square)],[Bakus]
8,cPH8TY_FP2nMGpFWXqpQb_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1060.823750,"[9368718200, 7041564226, 7041564225, 704156419...",[El-Mandara],[Bakus]
9,cURs_PxThe2b2znIGFxQs_Shape,-Q2gVX9GgVSEtVr-yv6Nf_Shape,1737.478490,"[7043828811, 7043828810, 9414993328, 702373236...",[Al-Rahma],[Bakus]


In [33]:
# Alternative visualization: draw raw GTFS shapes (lat/lon) + walking path
import folium

# Build raw shape polylines directly from shapes.txt order
shape_to_raw_coords = {}
for sid, pts in shape_to_points.items():
    # pts: list of (lat, lon, seq) in order
    shape_to_raw_coords[sid] = [(lat, lon) for (lat, lon, _) in pts]


def plot_shapes_raw_pair_with_path(start_shape_id, end_shape_id, save_html=None, map_tiles="cartodbpositron"):
    """
    Plot two raw GTFS shapes (no graph routing between shape points) and overlay the
    computed walking path between them from shape_level_pathways_df.
    """
    row = shape_level_pathways_df[(shape_level_pathways_df['start_shape_id'] == start_shape_id) & (shape_level_pathways_df['end_shape_id'] == end_shape_id)]
    if row.empty:
        raise ValueError("No pathway found for the given shape_id pair")
    row = row.iloc[0]
    walking_nodes = row['walking_path_nodes']

    # Center map at midpoint of walking path or first shape point
    if walking_nodes and len(walking_nodes) > 0:
        mid_n = walking_nodes[len(walking_nodes)//2]
        center_lat, center_lon = g.nodes[mid_n]['y'], g.nodes[mid_n]['x']
    else:
        c = shape_to_raw_coords.get(start_shape_id, [(31.2, 29.95)])
        center_lat, center_lon = c[0]

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    # Draw raw shapes
    c1 = shape_to_raw_coords.get(start_shape_id, [])
    c2 = shape_to_raw_coords.get(end_shape_id, [])

    if len(c1) >= 2:
        folium.PolyLine(c1, color="#1f77b4", weight=5, opacity=0.85, tooltip=f"shape {start_shape_id}").add_to(m)
    if len(c2) >= 2:
        folium.PolyLine(c2, color="#ff7f0e", weight=5, opacity=0.85, tooltip=f"shape {end_shape_id}").add_to(m)

    # Overlay walking path from graph nodes
    walk_coords = [(g.nodes[n]['y'], g.nodes[n]['x']) for n in walking_nodes if 'x' in g.nodes[n] and 'y' in g.nodes[n]]
    if len(walk_coords) >= 2:
        folium.PolyLine(walk_coords, color="#2ca02c", weight=6, opacity=0.95, tooltip=f"walk {len(walk_coords)} pts").add_to(m)
        sy, sx = walk_coords[0]
        ey, ex = walk_coords[-1]
        folium.CircleMarker([sy, sx], radius=5, color="#2ca02c", fill=True, tooltip="Walk start").add_to(m)
        folium.CircleMarker([ey, ex], radius=5, color="#2ca02c", fill=True, tooltip="Walk end").add_to(m)

    if save_html:
        m.save(save_html)
    return m

# Example:
# plot_shapes_raw_pair_with_path('gqntW1mb4ahLW_IHlR39R_Shape', 'udcnz_0kW2tOMy6xB2_Ng_Shape', save_html='raw_pair.html')


In [28]:
shape_level_pathways_df[shape_level_pathways_df['start_shape_id'] == 'gqntW1mb4ahLW_IHlR39R_Shape']

,start_shape_id,end_shape_id,walking_distance_m,walking_path_nodes,start_trip_names,end_trip_names
231,gqntW1mb4ahLW_IHlR39R_Shape,1pTTdCYNksTHRhSFGpg76_Shape,388.411565,"[1885907581, 1885907602, 6935289525, 243059732...",[Ezbet Saad],[El-Mansheya]
324,gqntW1mb4ahLW_IHlR39R_Shape,28mrnCSEb3wYAsdtd50rU_Shape,426.337782,"[1885907581, 1128126076, 8431986624, 694320049...",[Ezbet Saad],[Al-Wardiyan]
406,gqntW1mb4ahLW_IHlR39R_Shape,3DsaT0dcW0e0BLcyD3Q6E_Shape,383.619332,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[Kabo]
559,gqntW1mb4ahLW_IHlR39R_Shape,3SynmEGmJSDoBYRTRgEkj_Shape,296.954361,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[El-Mansheya]
968,gqntW1mb4ahLW_IHlR39R_Shape,6LBLdjAw8E6T2OCvlWEr8_Shape,208.123577,"[1885907581, 1885907602, 6935289525, 243059732...",[Ezbet Saad],[Green Plaza Mall]
1718,gqntW1mb4ahLW_IHlR39R_Shape,Cl-oT7WLLDtvd9UF3t3WF_Shape,409.600047,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[Anfoshy]
2007,gqntW1mb4ahLW_IHlR39R_Shape,E3cKTAI5hrMdq3gv0l-3g_Shape,383.619332,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[Al-Seyouf (Falaky)]
2059,gqntW1mb4ahLW_IHlR39R_Shape,EglZxh1NkJA5Jax303ARy_Shape,388.411565,"[1885907581, 1885907602, 6935289525, 243059732...",[Ezbet Saad],[El-Mansheya]
2138,gqntW1mb4ahLW_IHlR39R_Shape,EoBxLx54RiTmwf0L64PXs_Shape,383.619332,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[El-Mandara]
2631,gqntW1mb4ahLW_IHlR39R_Shape,J3ADe7YQWUaBHH8Z_00Ou_Shape,383.619332,"[1885907581, 1885907602, 6935289525, 243059731...",[Ezbet Saad],[El-Mandara]


In [35]:
plot_shapes_raw_pair_with_path(
    'gqntW1mb4ahLW_IHlR39R_Shape',
    'udcnz_0kW2tOMy6xB2_Ng_Shape',
    save_html='raw_pair.html'
)